# Hubble-8B unlearning — MASTER (baselines + aggregate + faithfulness)

Runs the **Standard/Perturbed baselines**, then pulls every method's results from HF and produces the comparison figures, the **rank-by-layer** logit lens, and the **LSR semantic-residual faithfulness** bake-off. Run AFTER the four method notebooks have pushed their models/results.

In [ ]:
# --- Setup: install deps, put the hubble8b package on the path, set HF token ---
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.53", "datasets", "accelerate", "bitsandbytes", "scipy"], check=False)

# Make `import hubble8b` work. Adjust HUB_DIR to wherever this repo lives on the box.
HUB_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
for cand in [HUB_DIR, os.path.join(HUB_DIR, "gpu_8b_run"), "/kaggle/input"]:
    if os.path.isdir(os.path.join(cand, "hubble8b")):
        sys.path.insert(0, cand)
        break

# HF token: use env in batch jobs; Colab userdata only when available.
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is not set. Export HF_TOKEN before submitting the job; nbconvert cannot prompt interactively.")

import hubble8b as h8
h8.hf_login()
print("device:", h8.DEVICE, "| target slot:", h8.TARGET_SLOT, "| LSR slot:", h8.LSR_SLOT)


In [ ]:
# --- Data: splits + retain + distractor pool (deterministic, seed=42) ---
forget_set, keep_set, unseen_set = h8.load_yago_splits()
retain = h8.load_retain_texts()
retain_texts = retain.select(range(min(len(retain), len(forget_set) * 40)))["text"]
splits = {"Unseen": unseen_set, "Unlearn": forget_set, "Keep": keep_set}
import json
pool_birth = h8.build_distractor_pool(forget_set, keep_set, unseen_set, h8.TARGET_SLOT)
print("birthdate distractors:", len(pool_birth))


## Baselines — Standard & Perturbed (infill), uploaded to HF

In [ ]:
import pandas as pd
for name in ["Standard", "Perturbed"]:
    m = h8.load_named(name)
    df = h8.run_infill(m, name, splits, pool_birth, target_slot=h8.TARGET_SLOT)
    p = f"{h8.workdir()}/infill_{name}.csv"; df.to_csv(p, index=False); h8.upload_artifact(p)
    h8.to_cpu(m); del m; h8.free_cuda()


## Aggregate infill accuracy (all models) — grouped bars

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
TAG = h8.config.SCALE_TAG
frames = []
for name in ["Standard","Perturbed","NPO","GradDiff","IDK","RMU"]:
    try:
        frames.append(pd.read_csv(h8.download_artifact(f"infill_{name}.csv")))
    except Exception as e:
        print(f"  {name}: no infill CSV ({type(e).__name__})")
if not frames:
    print("No infill CSVs found; skipping aggregate infill plot.")
else:
    inf = pd.concat(frames, ignore_index=True)
    order = ["Standard","Perturbed","NPO","GradDiff","IDK","RMU"]
    fig, axes = plt.subplots(2,2, figsize=(16,12)); axes=axes.flatten()
    for ax,fmt in zip(axes, h8.TEMPLATE_FORMATS):
        piv = inf[inf.template_format==fmt].pivot(index="split", columns="model", values="accuracy").reindex(["Unseen","Unlearn","Keep"])
        piv = piv[[m for m in order if m in piv.columns]]
        if piv.empty:
            ax.set_title(fmt)
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_axis_off()
            continue
        piv.plot(kind="bar", ax=ax, color=[h8.COLOR_MAP[m] for m in piv.columns], legend=(fmt==h8.TEMPLATE_FORMATS[0]))
        ax.set_title(fmt); ax.set_ylim(0,1); ax.axhline(0.1, ls="--", c="gray"); ax.grid(axis="y", alpha=0.3)
    fig.suptitle(f"Hubble-{TAG.upper()} / YAGO birthdate — infill accuracy", fontweight="bold"); fig.tight_layout()
    p=f"{h8.workdir()}/infill_bars_{TAG}.png"; fig.savefig(p,dpi=150,bbox_inches="tight"); h8.upload_artifact(p); plt.show()

## Rank-by-layer logit lens — forget vs retain (birthdate)

For each model, decode every hidden state through the final norm + LM head (`layerwise_target_rank`) and record the **rank of the gold token** at each layer, averaged over facts. Two panels share a log y-axis: **Forget (Unlearn)** and **Retain (Keep)**. A faithful unlearn pushes the gold rank *up* on the forget panel while keeping the retain panel *low* (near the Perturbed baseline). If the retain panel also inflates, the method damaged knowledge it was supposed to keep — which is exactly what the infill accuracies suggest here.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
tok = h8.get_tokenizer(); TAG = h8.config.SCALE_TAG
# Rank-by-layer logit lens on BOTH the forget (Unlearn) and retain (Keep) sets, TARGET_SLOT
# (birthdate). MEDIAN over facts (robust to heavy-tailed outliers); layerwise_target_rank uses
# the true final logits. RANK_FMT env chooses the probe context: "full_prefix" (default; 8B) or
# "full_prefix_full_suffix" (adds the memorized suffix span -> stronger recall at 1B where the
# model memorizes birthdate only ~half the time, otherwise the Perturbed baseline looks inverted).
RANK_FMT = os.environ.get("RANK_FMT", "full_prefix")
print("rank probe format:", RANK_FMT)
lens_sets = {"Forget (Unlearn)": forget_set, "Retain (Keep)": keep_set}
lens_facts = {k: h8.facts_for(v, slot=h8.TARGET_SLOT, fmt=RANK_FMT) for k, v in lens_sets.items()}
rank_curves = {k: {} for k in lens_sets}
for name in ["Standard","Perturbed","NPO","GradDiff","IDK","RMU"]:
    try:
        m = h8.load_named(name)
    except Exception as e:
        print(f"  {name}: unavailable ({type(e).__name__})")
        continue
    for set_name, facts in lens_facts.items():
        curves = []
        for prefix, gold in facts:
            try:
                pl = h8.layerwise_target_rank(m, tok, prefix, gold, h8.DEVICE)
            except Exception:
                pl = None
            if pl is None or not np.all(np.isfinite(pl)):
                continue
            curves.append(pl)
        if curves:
            rank_curves[set_name][name] = np.median(np.array(curves), axis=0)
            print(f"  [{name}] {set_name}: final-layer median rank={rank_curves[set_name][name][-1]:.1f} (n={len(curves)})")
        else:
            print(f"  [{name}] {set_name}: no rank samples")
    h8.to_cpu(m); del m; h8.free_cuda()
panels = [k for k in lens_sets if rank_curves[k]]
if not panels:
    print("No logit-lens samples available; skipping rank-by-layer plot.")
else:
    fig, axes = plt.subplots(1, len(panels), figsize=(8*len(panels), 6), sharey=True, squeeze=False)
    axes = axes[0]
    order = ["Standard","Perturbed","NPO","GradDiff","IDK","RMU"]
    for ax, set_name in zip(axes, panels):
        present = [m for m in order if m in rank_curves[set_name]]
        for nm in present:
            c = rank_curves[set_name][nm]
            ax.plot(range(len(c)), c, marker="o", ms=3, label=nm, color=h8.COLOR_MAP[nm])
        ax.set_yscale("log")
        ax.set_xlabel("hidden-state index (0 = embeddings, N = final layer)")
        ax.set_title(set_name)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("median rank of gold token (log; lower = recoverable)")
    axes[0].legend()
    fig.suptitle(f"Rank-by-layer logit lens — forget vs retain ({TAG.upper()}, {RANK_FMT})", fontweight="bold")
    fig.tight_layout()
    p=f"{h8.workdir()}/logitlens_rank_{TAG}.png"; fig.savefig(p,dpi=150,bbox_inches="tight"); h8.upload_artifact(p); plt.show()

## LSR — semantic-residual faithfulness (occupation), knowledge discrimination

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
tok = h8.get_tokenizer(); TAG = h8.config.SCALE_TAG
pert = h8.load_named("Perturbed"); E_REF = h8.build_Eref(pert).to(h8.DEVICE)
h8.to_cpu(pert); del pert; h8.free_cuda()
forget_facts = h8.facts_for(forget_set, slot=h8.LSR_SLOT)
lsr_sem, sem_curves = {}, {}
for name in ["Standard","Perturbed","NPO","GradDiff","IDK","RMU"]:
    try:
        m = h8.load_named(name)
    except Exception as e:
        print(f"  {name}: unavailable ({type(e).__name__})")
        continue
    sem, curves = [], []
    for prefix,gold in forget_facts:
        r = h8.layerwise_residual(m, tok, prefix, gold, E_REF, h8.DEVICE)
        if r is None:
            continue
        sem.append(h8.lsr_scalar(r[0])); curves.append(r[0])
    lsr_sem[name] = np.array(sem); sem_curves[name] = np.array(curves)
    if len(lsr_sem[name]):
        print(f"  [{name}] LSR mean={lsr_sem[name].mean():.4f}")
    else:
        print(f"  [{name}] no LSR samples")
    h8.to_cpu(m); del m; h8.free_cuda()
if "Perturbed" in lsr_sem and "Standard" in lsr_sem and len(lsr_sem["Perturbed"]) and len(lsr_sem["Standard"]):
    a, b = lsr_sem["Perturbed"], lsr_sem["Standard"]
    n = min(len(a), len(b))
    _, pv = wilcoxon(a[:n], b[:n])
    print(f"Discrimination (Perturbed vs Standard) p={pv:.2e}")
present = [m for m in ["Standard","Perturbed","NPO","GradDiff","IDK","RMU"] if m in lsr_sem and len(lsr_sem[m])]
if not present:
    print("No LSR samples available; skipping box plot.")
else:
    plt.figure(figsize=(10,5))
    # matplotlib >=3.9 removed the boxplot `labels=` kwarg; set tick labels explicitly.
    plt.boxplot([lsr_sem[m] for m in present], showmeans=True)
    plt.xticks(range(1, len(present) + 1), present)
    plt.ylabel("LSR (max C_f mass, mid/late) — higher = residual")
    plt.title(f"Semantic residual, forget set ({TAG.upper()})")
    plt.grid(axis="y", alpha=0.3); plt.tight_layout()
    p=f"{h8.workdir()}/lsr_boxplot_{TAG}.png"; plt.savefig(p,dpi=150,bbox_inches="tight"); h8.upload_artifact(p); plt.show()